In [58]:
import numpy as np
import pandas as pd
import re
import string 
import pickle

In [59]:
data = pd.read_csv('../artifacts/sentiment_analysis.csv')

In [60]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, '')
    return text

In [61]:
with open('../static/model/model.pickle','rb') as file:
    model = pickle.load(file)

In [62]:
with open('../static/model/corpora/stopwords/english','r') as file:
    sw = file.read().splitlines()

In [63]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [64]:
def preprocessing(text): 
    data = pd.DataFrame([text], columns=['tweet'])
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x.lower() for x in x.split()))
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*', '', word, flags=re.MULTILINE) for word in x.split()))
    data["tweet"] = data["tweet"].apply(remove_punctuations)
    data["tweet"] = data["tweet"].str.replace('\d+', '', regex=True)
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(x for x in x.split() if x not in sw))
    data["tweet"] = data["tweet"].apply(lambda x: " ".join(ps.stem(x) for x in x.split()))
    return data["tweet"]

In [65]:
preprocessed_txt = preprocessing(txt)

In [66]:
vocab = pd.read_csv('../static/model/vocabulary.txt', header=None)
token = vocab[0].tolist()

In [67]:

def vectorizer(ds, vocabulary):
    vectorized_list = []

    for sentence in ds:
        sentence_list = np.zeros(len(vocabulary))

        for i in range(len(vocabulary)):
            if vocabulary[i] in sentence.split():
                sentence_list[i] = 1

        vectorized_list.append(sentence_list)

    vectorized_lst_new = np.asarray(vectorized_list, dtype=np.float32)

    return np.array(vectorized_list)

In [68]:
vectorized_txt = vectorizer(preprocessed_txt,token)

In [69]:
def get_prediction(vectorized_text):
    prediction = model.predict(vectorized_text)
    if prediction == 1:
        return 'negetive'
    else:
        return 'possitive'

In [76]:
positive = [
    "good",
    "good product",
    "this is good",
    "very good",
    "nice and good"
]

In [77]:
txt = "awesome product. i love it"
preprocessed_txt = preprocessing(txt)
vectorized_txt = vectorizer(preprocessed_txt, token)
prediction = get_prediction(vectorized_txt)
prediction


'possitive'

In [78]:
txt = " "
preprocessed_txt = preprocessing(txt)
vectorized_txt = vectorizer(preprocessed_txt, token)
prediction = get_prediction(vectorized_txt)
prediction


'negetive'